# KASBench Analysis
## Download and Preprocessing

This notebook downloads the KASBench dataset from S3 and performs some basic preprocessing steps.



In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import io
import json
import os
import importlib

import boto3
import pandas as pd
import numpy as np
import sqlite3
import pyperclip
from dotenv import load_dotenv
import matplotlib.pyplot as plt

import common

from common import (
    get_s3_file_listing, 
    is_locust_db, 
    download_locust_db, 
    parse_locust_db_file_info,
    create_log_database,
    get_raw_log_as_dataframe,
    get_log_as_dataframe,
    get_run_details,
    get_autoscaler,
    get_trial_summary_df,
    get_autoscaler_summary,
    parse_roundtrip_file_info,
    merge_roundtrip_completion_percentage,
    )

importlib.reload(common)


### Get environmental variables

In [ ]:
load_dotenv()
s3_bucket = os.environ.get("S3_BUCKET")
print(f"S3 Bucket: {s3_bucket}")

### Set constants and create directories

In [ ]:
RUN = "exp-2026-08-02-c"
DATA_DIR = f"../data/{RUN}"
os.makedirs(DATA_DIR, exist_ok=True)
RUN_DB = f"{DATA_DIR}/logs.db"
print(f"Run database is {RUN_DB}")


### Get the file listing from S3   

In [ ]:
files = get_s3_file_listing(s3_bucket, s3_prefix=RUN)


### Iterate the file listing and download all the Locust databases to local storage

In [ ]:
# Delete the log database if it already exists
if os.path.exists(RUN_DB):
    os.remove(RUN_DB)

# Create the log database
create_log_database(RUN_DB)

# Populate the log database from the locust database files
for file_info in files:
    # skip non-database files
    if not is_locust_db(file_info):
        continue
    
    # parse trial, role, and filename from the S3 file info
    trial, role, filename = parse_locust_db_file_info(file_info)

    # get the autoscaler from the run details
    run_details = get_run_details(s3_bucket, RUN, trial)
    autoscaler = get_autoscaler(run_details)

    # skip if it-operations.  It operations transactions are not tracked for response time/failure rate
    if role == "it-operations":
        continue
    
    # download the database if it doesn't already exist
    local_path = download_locust_db(DATA_DIR, s3_bucket, file_info["Key"], trial, role, filename)
    
    # load as a dataframe, augmented with run id, trial id, and role
    df = get_raw_log_as_dataframe(local_path)
    df["run_id" ] = RUN
    df["trial_id" ] = trial
    df["role" ] = role
    df["autoscaler" ] = autoscaler
    df["success"] = (
        pd.to_numeric(df["status_code"], errors="coerce")
          .between(200, 299)
        )
    df["failure"] = ~df["success"]

    # append to the sqlite database
    df.to_sql('logs', sqlite3.connect(RUN_DB), if_exists='append', index=False)
    


    


### Load the aggregate database into a dataframe for analysis

In [ ]:
df = get_log_as_dataframe(RUN_DB)
raw_df = df.copy()
df

### Summarize the trials

In [ ]:
trial_summary_df = get_trial_summary_df(df)
trial_summary_df

### Merge Roundtrip Percentage

In [ ]:
merged_trial_summary_df = merge_roundtrip_completion_percentage(files, s3_bucket, trial_summary_df)
merged_trial_summary_df


In [ ]:
merged_trial_summary_df.columns

### Aggregate the summarized trials

In [ ]:
autoscaler_summary = get_autoscaler_summary(merged_trial_summary_df)
autoscaler_summary

### Visualize the results

#### Plot mean response time by autoscaler


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))

autoscalers = sorted(merged_trial_summary_df["autoscaler"].unique())

for i, a in enumerate(autoscalers):
    d = trial_summary_df[trial_summary_df.autoscaler == a]

    # jittered trial means
    x = np.random.normal(i, 0.04, len(d))
    ax.scatter(x, d["mean_response_time"],
               alpha=0.7, s=40)

    mean = d["mean_response_time"].mean()
    se = d["mean_response_time"].std(ddof=1) / np.sqrt(len(d))

    ax.errorbar(
        i,
        mean,
        yerr=1.96 * se,
        fmt="o",
        capsize=6,
        markersize=8,
        linewidth=2,
    )

ax.set_xticks(range(len(autoscalers)))
ax.set_xticklabels(autoscalers)
ax.set_ylabel("Mean response time (ms)")
ax.set_xlabel("Autoscaler")

plt.tight_layout()

In [ ]:
fig.savefig(
    "../figures/response_time_by_autoscaler.png",
    dpi=300,
    bbox_inches="tight",
)

#### Plot failure rate by autoscaler


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import t


def plot_failure_rate_by_autoscaler(
    trial_summary: pd.DataFrame,
    *,
    autoscaler_order: list[str] | None = None,
    confidence_level: float = 0.95,
    random_seed: int = 42,
) -> tuple[plt.Figure, plt.Axes]:
    """
    Plot trial-level failure rates by autoscaler.

    Each small point represents one trial. The larger point represents the
    unweighted mean of the trial failure rates. Error bars show a two-sided
    Student-t confidence interval for the mean.

    Parameters
    ----------
    trial_summary:
        DataFrame containing:
          - autoscaler
          - trial_id
          - failure_rate

        failure_rate must be expressed as a proportion, e.g. 0.025 for 2.5%.

    autoscaler_order:
        Optional display order. Autoscalers not listed here are appended
        alphabetically.

    confidence_level:
        Confidence level for the error bars. Default is 0.95.

    random_seed:
        Seed used to make horizontal point jitter reproducible.

    Returns
    -------
    tuple[matplotlib.figure.Figure, matplotlib.axes.Axes]
        The generated figure and axes.
    """
    required_columns = {"autoscaler", "trial_id", "failure_rate"}
    missing_columns = required_columns - set(trial_summary.columns)

    if missing_columns:
        raise ValueError(
            f"trial_summary is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if not 0 < confidence_level < 1:
        raise ValueError("confidence_level must be between 0 and 1.")

    plot_df = trial_summary[
        ["autoscaler", "trial_id", "failure_rate"]
    ].copy()

    plot_df["failure_rate"] = pd.to_numeric(
        plot_df["failure_rate"],
        errors="coerce",
    )

    if plot_df["failure_rate"].isna().any():
        bad_rows = plot_df.loc[plot_df["failure_rate"].isna()]
        raise ValueError(
            "failure_rate contains missing or nonnumeric values. "
            f"Invalid rows:\n{bad_rows}"
        )

    if not plot_df["failure_rate"].between(0, 1).all():
        bad_rows = plot_df.loc[
            ~plot_df["failure_rate"].between(0, 1)
        ]
        raise ValueError(
            "failure_rate must be between 0 and 1. "
            f"Invalid rows:\n{bad_rows}"
        )

    present_autoscalers = sorted(
        plot_df["autoscaler"].dropna().unique()
    )

    if autoscaler_order is None:
        autoscalers = present_autoscalers
    else:
        autoscalers = [
            autoscaler
            for autoscaler in autoscaler_order
            if autoscaler in present_autoscalers
        ]

        autoscalers.extend(
            autoscaler
            for autoscaler in present_autoscalers
            if autoscaler not in autoscalers
        )

    rng = np.random.default_rng(random_seed)

    fig, ax = plt.subplots(figsize=(8, 5.5))

    alpha = 1 - confidence_level
    summary_rows = []

    for position, autoscaler in enumerate(autoscalers):
        autoscaler_df = plot_df.loc[
            plot_df["autoscaler"] == autoscaler
        ]

        rates_percent = (
            autoscaler_df["failure_rate"].to_numpy() * 100
        )

        # Add slight horizontal jitter so overlapping trial points remain visible.
        jittered_x = rng.normal(
            loc=position,
            scale=0.045,
            size=len(rates_percent),
        )

        ax.scatter(
            jittered_x,
            rates_percent,
            s=45,
            alpha=0.65,
            label="Individual trial" if position == 0 else None,
            zorder=2,
        )

        number_of_trials = len(rates_percent)
        mean_rate = rates_percent.mean()

        if number_of_trials >= 2:
            standard_deviation = rates_percent.std(ddof=1)
            standard_error = (
                standard_deviation / np.sqrt(number_of_trials)
            )

            critical_value = t.ppf(
                1 - alpha / 2,
                df=number_of_trials - 1,
            )

            confidence_interval_half_width = (
                critical_value * standard_error
            )
        else:
            standard_deviation = np.nan
            standard_error = np.nan
            confidence_interval_half_width = np.nan

        ax.errorbar(
            position,
            mean_rate,
            yerr=confidence_interval_half_width,
            fmt="o",
            markersize=9,
            capsize=7,
            capthick=1.5,
            linewidth=2,
            label=(
                f"Mean and {confidence_level:.0%} CI"
                if position == 0
                else None
            ),
            zorder=3,
        )

        summary_rows.append(
            {
                "autoscaler": autoscaler,
                "trials": number_of_trials,
                "mean_failure_rate": mean_rate / 100,
                "sd_failure_rate": standard_deviation / 100,
                "se_failure_rate": standard_error / 100,
                "ci_lower": (
                    max(
                        0,
                        mean_rate
                        - confidence_interval_half_width,
                    )
                    / 100
                    if number_of_trials >= 2
                    else np.nan
                ),
                "ci_upper": (
                    min(
                        100,
                        mean_rate
                        + confidence_interval_half_width,
                    )
                    / 100
                    if number_of_trials >= 2
                    else np.nan
                ),
            }
        )

    ax.set_xticks(range(len(autoscalers)))
    ax.set_xticklabels(
        [autoscaler.upper() for autoscaler in autoscalers]
    )

    ax.set_xlabel("Autoscaler")
    ax.set_ylabel("Failure rate (%)")
    ax.set_title(
        "Failure Rate by Autoscaler\n"
        "Trial-level rates with mean and "
        f"{confidence_level:.0%} confidence interval"
    )

    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

    fig.tight_layout()

    # The summary can be retrieved from the axes for later use if desired.
    ax.failure_rate_summary = pd.DataFrame(summary_rows)

    return fig, ax

In [ ]:
fig, ax = plot_failure_rate_by_autoscaler(
    merged_trial_summary_df,
    autoscaler_order=["none", "hpa", "vpa", "keda"],
)

plt.show()

In [ ]:
fig.savefig(
    "../figures/failure_rate_by_autoscaler.png",
    dpi=300,
    bbox_inches="tight",
)

#### Plot of roundtrip completion percentage by autoscaler

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import t


def plot_roundtrip_completion_by_autoscaler(
    trial_summary: pd.DataFrame,
    *,
    autoscaler_order: list[str] | None = None,
    confidence_level: float = 0.95,
    random_seed: int = 42,
) -> tuple[plt.Figure, plt.Axes]:
    """
    Plot trial-level round-trip completion percentages by autoscaler.

    Each small point represents one trial. The larger point represents the
    unweighted mean of the trial percentages. Error bars show a two-sided
    Student-t confidence interval for the mean.

    Parameters
    ----------
    trial_summary:
        DataFrame containing:
          - autoscaler
          - trial_id
          - roundtrip_completion_percentage

        roundtrip_completion_percentage must be expressed in percentage
        points, such as 9.3 for 9.3%, rather than 0.093.

    autoscaler_order:
        Optional display order. Autoscalers present in the dataframe but not
        listed here are appended alphabetically.

    confidence_level:
        Confidence level for the error bars. Defaults to 0.95.

    random_seed:
        Seed used to make the horizontal jitter reproducible.

    Returns
    -------
    tuple[matplotlib.figure.Figure, matplotlib.axes.Axes]
        The generated figure and axes.
    """
    required_columns = {
        "autoscaler",
        "trial_id",
        "roundtrip_completion_percentage",
    }
    missing_columns = required_columns - set(trial_summary.columns)

    if missing_columns:
        raise ValueError(
            "trial_summary is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if not 0 < confidence_level < 1:
        raise ValueError("confidence_level must be between 0 and 1.")

    plot_df = trial_summary[
        [
            "autoscaler",
            "trial_id",
            "roundtrip_completion_percentage",
        ]
    ].copy()

    if plot_df["autoscaler"].isna().any():
        raise ValueError("autoscaler contains missing values.")

    if plot_df["trial_id"].isna().any():
        raise ValueError("trial_id contains missing values.")

    plot_df["roundtrip_completion_percentage"] = pd.to_numeric(
        plot_df["roundtrip_completion_percentage"],
        errors="coerce",
    )

    if plot_df["roundtrip_completion_percentage"].isna().any():
        bad_rows = plot_df.loc[
            plot_df["roundtrip_completion_percentage"].isna()
        ]
        raise ValueError(
            "roundtrip_completion_percentage contains missing or "
            f"nonnumeric values:\n{bad_rows}"
        )

    if not plot_df["roundtrip_completion_percentage"].between(
        0, 100
    ).all():
        bad_rows = plot_df.loc[
            ~plot_df["roundtrip_completion_percentage"].between(0, 100)
        ]
        raise ValueError(
            "roundtrip_completion_percentage must be between 0 and 100. "
            f"Invalid rows:\n{bad_rows}"
        )

    # Prevent a trial from being counted more than once for an autoscaler.
    duplicate_mask = plot_df.duplicated(
        subset=["autoscaler", "trial_id"],
        keep=False,
    )

    if duplicate_mask.any():
        duplicate_rows = plot_df.loc[
            duplicate_mask,
            ["autoscaler", "trial_id"],
        ].sort_values(["autoscaler", "trial_id"])

        raise ValueError(
            "Each autoscaler/trial_id combination must appear exactly once. "
            f"Duplicate rows:\n{duplicate_rows}"
        )

    present_autoscalers = sorted(
        plot_df["autoscaler"].astype(str).unique()
    )

    if autoscaler_order is None:
        autoscalers = present_autoscalers
    else:
        autoscalers = [
            autoscaler
            for autoscaler in autoscaler_order
            if autoscaler in present_autoscalers
        ]

        autoscalers.extend(
            autoscaler
            for autoscaler in present_autoscalers
            if autoscaler not in autoscalers
        )

    if not autoscalers:
        raise ValueError("No autoscaler data is available to plot.")

    rng = np.random.default_rng(random_seed)
    alpha = 1 - confidence_level

    fig, ax = plt.subplots(figsize=(8, 5.5))

    summary_rows: list[dict[str, float | int | str]] = []

    for position, autoscaler in enumerate(autoscalers):
        autoscaler_df = plot_df.loc[
            plot_df["autoscaler"] == autoscaler
        ].sort_values("trial_id")

        percentages = autoscaler_df[
            "roundtrip_completion_percentage"
        ].to_numpy(dtype=float)

        number_of_trials = len(percentages)

        # Slight horizontal jitter keeps overlapping trial points visible.
        jittered_x = rng.normal(
            loc=position,
            scale=0.045,
            size=number_of_trials,
        )

        ax.scatter(
            jittered_x,
            percentages,
            s=45,
            alpha=0.65,
            label="Individual trial" if position == 0 else None,
            zorder=2,
        )

        mean_percentage = percentages.mean()

        if number_of_trials >= 2:
            standard_deviation = percentages.std(ddof=1)
            standard_error = (
                standard_deviation / np.sqrt(number_of_trials)
            )

            critical_value = t.ppf(
                1 - alpha / 2,
                df=number_of_trials - 1,
            )

            confidence_interval_half_width = (
                critical_value * standard_error
            )

            confidence_interval_lower = max(
                0.0,
                mean_percentage - confidence_interval_half_width,
            )
            confidence_interval_upper = min(
                100.0,
                mean_percentage + confidence_interval_half_width,
            )

            # Asymmetric errors allow confidence bounds to be clipped to
            # the logically valid 0%-100% interval.
            lower_error = mean_percentage - confidence_interval_lower
            upper_error = confidence_interval_upper - mean_percentage
            yerr = np.array([[lower_error], [upper_error]])
        else:
            standard_deviation = np.nan
            standard_error = np.nan
            critical_value = np.nan
            confidence_interval_half_width = np.nan
            confidence_interval_lower = np.nan
            confidence_interval_upper = np.nan
            yerr = None

        ax.errorbar(
            position,
            mean_percentage,
            yerr=yerr,
            fmt="o",
            markersize=9,
            capsize=7,
            capthick=1.5,
            linewidth=2,
            label=(
                f"Mean and {confidence_level:.0%} CI"
                if position == 0
                else None
            ),
            zorder=3,
        )

        summary_rows.append(
            {
                "autoscaler": autoscaler,
                "trials": number_of_trials,
                "mean_roundtrip_completion_percentage": mean_percentage,
                "sd_roundtrip_completion_percentage": standard_deviation,
                "se_roundtrip_completion_percentage": standard_error,
                "t_critical": critical_value,
                "ci_half_width": confidence_interval_half_width,
                "ci_lower": confidence_interval_lower,
                "ci_upper": confidence_interval_upper,
            }
        )

    ax.set_xticks(range(len(autoscalers)))
    ax.set_xticklabels(
        [autoscaler.upper() for autoscaler in autoscalers]
    )

    ax.set_xlabel("Autoscaler")
    ax.set_ylabel("Round-trip completion percentage (%)")
    ax.set_title(
        "Round-Trip Completion by Autoscaler\n"
        "Trial-level percentages with mean and "
        f"{confidence_level:.0%} confidence interval"
    )

    # Start at zero because the metric has a meaningful zero.
    # Let Matplotlib choose the upper bound unless the values approach 100%.
    ax.set_ylim(bottom=0)

    ax.grid(
        axis="y",
        alpha=0.25,
    )
    ax.legend()

    fig.tight_layout()

    # Attach the statistics used in the chart for convenient retrieval.
    ax.roundtrip_completion_summary = pd.DataFrame(summary_rows)

    return fig, ax

In [ ]:
fig, ax = plot_roundtrip_completion_by_autoscaler(
    merged_trial_summary_df,
    autoscaler_order=["none", "hpa", "vpa", "keda"],
)

plt.show()

In [ ]:
fig.savefig(
    "../figures/roundtrip_completion_by_autoscaler.png",
    dpi=300,
    bbox_inches="tight",
)

## Calculate $\theta$ and $\tau$

In [ ]:
TIME_SLICE_SECONDS = 30

In [ ]:
raw_df()